# Data Mining TA — Improvement

Notebook ini berisi versi perbaikan dari model prediksi tren pasar UMKM Bogor dengan:
1. **Stratified K-Fold Cross Validation** — evaluasi lebih stabil
2. **Cek distribusi label (class balance)**
3. **Hyperparameter Tuning** dengan `GridSearchCV`
4. **Evaluasi lebih lengkap** (Precision, Recall, F1, AUC-ROC per fold)

## 1. Install & Import Library

In [ ]:
!pip install Sastrawi -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import re
import warnings
warnings.filterwarnings('ignore')

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    cross_validate
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_auc_score,
    roc_curve,
    f1_score,
    precision_score,
    recall_score
)

sns.set_theme(style='whitegrid')
print('✅ Library berhasil diimport!')

## 2. Load Dataset

In [ ]:
# ⚠️ Ganti path sesuai lokasi file di Google Drive kamu
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/MBKM/flask_api/dataset_umkm_bogor.csv')

print(f'Jumlah data: {len(df)}')
print(f'Kolom: {list(df.columns)}')
df.head()

## 3. Text Preprocessing

In [ ]:
stemmer = StemmerFactory().create_stemmer()

stopwords = {
    'murah','promo','cod','terlaris','original','ori','asli',
    'oleh','pcs','gr','gram','kg','dan','di','ke','dari','yang'
}

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stopwords]
    return stemmer.stem(' '.join(words))

print('Memproses teks... (mungkin butuh beberapa menit)')
df['nama_produk_clean'] = df['nama_produk'].apply(clean_text)
print('✅ Selesai!')

## 4. Dynamic Labeling & Cek Distribusi Kelas

In [ ]:
median_terjual = df['jumlah_terjual'].median()
df['label'] = (df['jumlah_terjual'] > median_terjual).astype(int)

print(f'Median jumlah terjual: {median_terjual}')
print(f"\nDistribusi Label:")
print(df['label'].value_counts())
print(f"\nRasio kelas:")
print(df['label'].value_counts(normalize=True).round(3))

In [ ]:
# Visualisasi distribusi kelas
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Distribusi Kelas')
axes[0].set_xlabel('Label (0=Kurang Menarik, 1=Menarik)')
axes[0].set_ylabel('Jumlah Data')
axes[0].set_xticklabels(['Kurang Menarik (0)', 'Menarik (1)'], rotation=0)

# Jumlah terjual
axes[1].hist(df['jumlah_terjual'], bins=50, color='steelblue', edgecolor='white')
axes[1].axvline(x=median_terjual, color='red', linestyle='--', label=f'Median ({median_terjual})')
axes[1].set_title('Distribusi Jumlah Terjual')
axes[1].set_xlabel('Jumlah Terjual')
axes[1].set_ylabel('Frekuensi')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Feature Engineering & Persiapan Data

In [ ]:
text_feature = 'nama_produk_clean'
cat_features = ['kategori', 'sub_kategori']
num_features = ['harga_produk', 'rating']

preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(max_features=1000), text_feature),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
        ('num', StandardScaler(), num_features)
    ]
)

X = df[[text_feature] + cat_features + num_features]
y = df['label']

# Split untuk evaluasi akhir (holdout set)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Total data         : {len(X)}')
print(f'Train+Val set      : {len(X_train_val)}')
print(f'Test set (holdout) : {len(X_test)}')

## 6. Evaluasi dengan Stratified K-Fold Cross Validation

> **Mengapa K-Fold?** Dengan dataset kecil (n=597), hasil evaluasi dari satu train/test split bisa kurang stabil. K-Fold membagi data menjadi K bagian dan mengevaluasi model K kali, sehingga hasil lebih representatif.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Pipeline model Random Forest (baseline — sebelum tuning)
rf_baseline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Cross validation dengan berbagai metrik
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

print('Menjalankan 5-Fold Cross Validation untuk Random Forest (baseline)...')
cv_results_rf = cross_validate(
    rf_baseline, X_train_val, y_train_val,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

print('\n=== HASIL CROSS VALIDATION — RANDOM FOREST (BASELINE) ===')
print(f"{'Metrik':<15} {'Mean':>8} {'Std':>8}")
print('-' * 35)
for metric in scoring:
    scores = cv_results_rf[f'test_{metric}']
    print(f"{metric:<15} {scores.mean():.4f}   ±{scores.std():.4f}")

In [ ]:
# Juga evaluasi Logistic Regression untuk perbandingan
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

print('Menjalankan 5-Fold Cross Validation untuk Logistic Regression...')
cv_results_lr = cross_validate(
    lr_pipeline, X_train_val, y_train_val,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

print('\n=== PERBANDINGAN CROSS VALIDATION ===')
print(f"{'Metrik':<15} {'LR Mean':>10} {'RF Mean':>10} {'Winner':>10}")
print('-' * 50)
for metric in scoring:
    lr_mean = cv_results_lr[f'test_{metric}'].mean()
    rf_mean = cv_results_rf[f'test_{metric}'].mean()
    winner = 'RF ✅' if rf_mean >= lr_mean else 'LR ✅'
    print(f"{metric:<15} {lr_mean:>10.4f} {rf_mean:>10.4f} {winner:>10}")

In [ ]:
# Visualisasi hasil K-Fold
metrics_to_plot = ['accuracy', 'f1', 'roc_auc']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, metric in enumerate(metrics_to_plot):
    lr_scores = cv_results_lr[f'test_{metric}']
    rf_scores = cv_results_rf[f'test_{metric}']

    data = pd.DataFrame({
        'Logistic Regression': lr_scores,
        'Random Forest': rf_scores
    })

    data.boxplot(ax=axes[i])
    axes[i].set_title(f'{metric.upper()} — 5-Fold CV')
    axes[i].set_ylabel(metric)
    axes[i].set_ylim([0, 1])

plt.suptitle('Perbandingan Model — Stratified 5-Fold Cross Validation', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 7. Hyperparameter Tuning Random Forest

> **Mengapa tuning?** Hyperparameter default tidak selalu optimal. `GridSearchCV` mencoba berbagai kombinasi dan memilih yang terbaik berdasarkan AUC-ROC.

In [ ]:
# Grid parameter yang akan dicoba
param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf': [1, 2]
}

rf_for_grid = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

print('Menjalankan GridSearchCV (ini mungkin memakan waktu 5-15 menit)...')
grid_search = GridSearchCV(
    rf_for_grid,
    param_grid,
    cv=skf,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_val, y_train_val)

print('\n✅ Selesai!')
print(f'\nBest Parameters: {grid_search.best_params_}')
print(f'Best AUC-ROC (CV): {grid_search.best_score_:.4f}')

In [ ]:
# Simpan model terbaik
best_rf = grid_search.best_estimator_

# Evaluasi di test set (holdout)
y_pred_best = best_rf.predict(X_test)
y_proba_best = best_rf.predict_proba(X_test)[:, 1]

print('=== EVALUASI MODEL TERBAIK (HOLDOUT TEST SET) ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_best):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_best):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred_best):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred_best):.4f}')
print(f'AUC-ROC  : {roc_auc_score(y_test, y_proba_best):.4f}')
print()
print(classification_report(y_test, y_pred_best, target_names=['Kurang Menarik', 'Menarik']))

## 8. Perbandingan Lengkap: Baseline vs Tuned

In [ ]:
# Fit model baseline di seluruh train_val set
lr_pipeline.fit(X_train_val, y_train_val)
rf_baseline.fit(X_train_val, y_train_val)

# Prediksi semua model
models = {
    'Logistic Regression': lr_pipeline,
    'Random Forest (baseline)': rf_baseline,
    'Random Forest (tuned)': best_rf
}

print(f"{'Model':<30} {'Accuracy':>10} {'F1':>8} {'AUC-ROC':>10}")
print('-' * 62)
for name, model in models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    print(f"{name:<30} {acc:>10.4f} {f1:>8.4f} {auc:>10.4f}")

In [ ]:
# ROC Curve semua model
plt.figure(figsize=(8, 6))

colors = ['#3498db', '#e74c3c', '#2ecc71']
for (name, model), color in zip(models.items(), colors):
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Perbandingan Semua Model')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix model terbaik
cm = confusion_matrix(y_test, best_rf.predict(X_test))
TN, FP, FN, TP = cm.ravel()

labels = np.array([
    [f'TN\n{TN}', f'FP\n{FP}'],
    [f'FN\n{FN}', f'TP\n{TP}']
])

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm, annot=labels, fmt='', cmap='Greens',
    xticklabels=['Kurang Menarik', 'Menarik'],
    yticklabels=['Kurang Menarik', 'Menarik']
)
plt.title('Confusion Matrix — Random Forest (Tuned)')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.tight_layout()
plt.show()

## 9. Feature Importance (Top 20 Kata)

In [ ]:
# Ambil nama fitur dari preprocessor
tfidf_features = best_rf.named_steps['preprocessor'].transformers_[0][1].get_feature_names_out()
cat_features_names = best_rf.named_steps['preprocessor'].transformers_[1][1].get_feature_names_out()
num_features_names = np.array(num_features)

all_features = np.concatenate([tfidf_features, cat_features_names, num_features_names])

# Ambil importances dari RF
importances = best_rf.named_steps['classifier'].feature_importances_

# Top 20 fitur
top_idx = np.argsort(importances)[-20:][::-1]
top_features = all_features[top_idx]
top_importances = importances[top_idx]

plt.figure(figsize=(10, 6))
bars = plt.barh(range(20), top_importances[::-1], color='steelblue')
plt.yticks(range(20), top_features[::-1])
plt.xlabel('Feature Importance')
plt.title('Top 20 Fitur yang Paling Berpengaruh')
plt.tight_layout()
plt.show()

print('Top 20 Fitur:')
for feat, imp in zip(top_features, top_importances):
    print(f'  {feat:<30} {imp:.4f}')

## 10. Simpan Model Terbaik

In [ ]:
# Retrain model terbaik dengan SELURUH data (train+val+test)
# untuk deployment production

best_params = grid_search.best_params_

final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=best_params['classifier__n_estimators'],
        max_depth=best_params['classifier__max_depth'],
        min_samples_split=best_params['classifier__min_samples_split'],
        min_samples_leaf=best_params['classifier__min_samples_leaf'],
        random_state=42
    ))
])

final_model.fit(X, y)  # Fit ke seluruh dataset

# Simpan ke Google Drive
save_path = '/content/drive/MyDrive/MBKM/flask_api/model_umkm_bogor_improved.joblib'
joblib.dump(final_model, save_path)

print(f'✅ Model berhasil disimpan ke: {save_path}')
print(f'\nBest Hyperparameters:')
for k, v in best_params.items():
    print(f'  {k}: {v}')

## 11. Ringkasan Improvement

| Aspek | Sebelumnya | Sesudah |
|-------|-----------|--------|
| **Evaluasi** | 1x train/test split | 5-Fold Cross Validation |
| **Hyperparameter** | Default (n_estimators=100) | GridSearchCV (optimal) |
| **Metrik** | Acc, AUC | Acc, Precision, Recall, F1, AUC |
| **Stabilitas** | Bergantung 1 split | Rata-rata 5 fold = lebih stabil |
| **Feature Importance** | Tidak ada | Ditampilkan (interpretabilitas) |